# Chapitre 6 — Risques garantis (selective classification)

Un score `P(malin)` ne dit rien sur le **risque réel** d'erreur. Le papier *Beyond
Accuracy: Controlling Broad Error Types in Selective Classification* (Jemelen et
al., AISTATS 2026) fournit des bornes **distribution-free** : on choisit un seuil
de confiance θ, on **s'abstient** sous ce seuil, et on obtient une **garantie**
(ex. « ≤ 2 % d'erreur, avec probabilité 99,5 % ») sur les cas retenus, en échange
d'une certaine **couverture** (proportion de cas non abstenus).

Le code est cloné dans `~/selective-classification`. Ce chapitre tourne sur CPU.
On utilise ici un **jeu synthétique** reproduisant les conditions d'un dépistage
(~2 % de cancers) pour montrer **ce qui est garantissable et ce qui ne l'est pas**.

In [ ]:
import os, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.expanduser('~/selective-classification'))
from python_scripts.math_utils import B_star
from python_scripts.sgp_utils import sgp_at_targets, emp_metric, DELTA
from python_scripts.preprocessing import train_test_split as sgp_split

print('DELTA par défaut =', DELTA, '(soit', 100 * (1 - DELTA), '% de confiance)')
# B_star = borne haute de Clopper-Pearson (Proposition A1) : 0 erreur sur n -> borne
print('B_star(delta=0.005, e=0, n=200) =', round(B_star(0.005, 0, 200), 4))

## Un jeu de prédictions réaliste

On simule un classifieur de dépistage : prévalence **2 %**, des scores plus élevés
pour les vrais cancers. On en tire les colonnes attendues par la librairie :

- `y_true` : vrai label (0/1) ;
- `y_pred` : décision au seuil 0,5 ;
- `kappa` : **confiance** = `max(p, 1-p)` ∈ [0,5 ; 1] (sert à abstenir).

In [ ]:
rng = np.random.default_rng(0)
n = 12000
y_true = (rng.random(n) < 0.02).astype(int)            # ~2 % de cancers
p = np.clip(rng.normal(np.where(y_true == 1, 0.62, 0.14), 0.18), 1e-3, 1 - 1e-3)
y_pred = (p >= 0.5).astype(int)
kappa = np.maximum(p, 1 - p)
df = pd.DataFrame({'kappa': kappa, 'y_pred': y_pred, 'y_true': y_true})

print('prévalence :', round(df.y_true.mean(), 4), '| positifs :', int(df.y_true.sum()))
print('Sur TOUT le set (sans abstention) :')
for m in ['standard', 'FPR', 'FNR']:
    print(f'  {m:9s} = {emp_metric(df, m):.3f}')
print('-> au seuil 0,5, le modèle rate beaucoup de cancers (FNR élevé) mais alarme peu (FPR faible)')

## Garantir le risque global (0/1)

`sgp_at_targets` cherche, pour chaque cible, le seuil θ le plus bas (couverture
max) tel que la **borne** sur le risque (pas seulement l'empirique) reste sous la
cible, avec confiance 1−δ. On coupe le jeu en *bound-fitting* (train) / *test*.

In [ ]:
train, test = sgp_split(df, seed=0, p_train=0.5)
res_std = sgp_at_targets(train, test, delta=5e-3,
                         metric_targets=[0.01, 0.02, 0.03, 0.05],
                         metric='standard', mode='dicho')
cols = ['metric_target', 'metric_bound', 'theta_star', 'test_metric', 'test_coverage']
print(res_std[cols].round(4).to_string(index=False))
print('\n-> « erreur 0/1 garantie ≤ 2 % à 99,5 % » est atteignable, en abstenant un peu (couverture < 1).')

## Le point clé : **FPR garantissable, FNR non**

FPR et FNR sont le **même objet** :

$$\text{borne} = \frac{b}{d_1 - d_2}, \quad d_2 = \frac{\sqrt{n\log(2/\delta)/2}}{|\text{sélection}|}$$

où `d1` est la proportion de la **classe conditionnée** : les **sains** pour FPR
(~98 % → `d1`≫`d2` → borne fine), les **cancers** pour FNR (~2 % → `d1`≈`d2` →
dénominateur ≈ 0 → borne **hors [0,1]**, donc inutile). Augmenter `n` à prévalence
fixe ne sépare pas `d1` et `d2`. Vérifions-le.

In [ ]:
targets = [0.05, 0.10, 0.20]
out = {}
for metric in ['FPR', 'FNR']:
    try:
        r = sgp_at_targets(train, test, delta=5e-3, metric_targets=targets,
                           metric=metric, mode='greedy')
        out[metric] = r
        print(f'=== {metric} ===')
        print(r[['metric_target', 'metric_bound', 'theta_star', 'test_metric', 'test_coverage']].round(4).to_string(index=False))
    except Exception as e:
        print(f'=== {metric} === échec :', e)
    print()

In [ ]:
# Visualisation : une borne n'est utile que si elle est < 1
fig, ax = plt.subplots(figsize=(7, 4))
for metric, color in [('FPR', '#2b8a3e'), ('FNR', '#c92a2a')]:
    if metric in out:
        r = out[metric]
        ax.plot(r['metric_target'], r['metric_bound'].clip(upper=2), 'o-', color=color, label=metric)
ax.axhline(1.0, ls='--', color='grey')
ax.text(targets[0], 1.02, 'borne = 1 : inutile (vacuous)', color='grey', fontsize=9)
ax.set_xlabel('cible'); ax.set_ylabel('borne garantie'); ax.legend()
ax.set_title('FPR (sains abondants) garantissable, FNR (cancers rares) non')
plt.tight_layout(); plt.show()

## Conclusion (honnête)

- ✅ **Côté négatif, ça marche** : risque global 0/1, FP, **FPR**, spécificité se
  garantissent à 99,5 %, en abstenant peu — parce que les sains sont abondants.
- ❌ **La sensibilité garantie ne marche PAS** sur un dépistage normal (~2 %) :
  **FNR / SE / PPV ne sont pas garantissables**, quel que soit δ. Ce n'est pas un
  défaut du modèle mais une **limite structurelle** : avec si peu de cancers, le
  terme de concentration `d2` rattrape la prévalence `d1`.
- 🔧 **Seul levier** : changer la **prévalence** du jeu de calibration
  (sous-échantillonner les sains / enrichir en cancers) — et même alors la
  sensibilité garantie plafonne (≈ 0,78 à 95 % avec ~246 cancers).

La valeur de l'abstention ici n'est pas un gain de performance observé, mais une
**garantie mathématiquement prouvée**, distribution-free — à condition de viser les
métriques que les données permettent réellement de borner.